# Project Description
This is under construction and will be updated soon

## Initial Setup

- Load `Qwen3.5-4B` from **unsloth** and initialize the weights
- Configure the **LoRA** adapters for the model
- Load and explore the dataset `lavita/MedQuad` from **HuggingFace**

In [3]:
from unsloth import FastVisionModel

# Initialize model weights and tokenizer
model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/qwen3.5-4B",
    load_in_4bit=False, # Qwen3.5 does not support 4-bit quantization; QLoRA not possible; use only LoRA 
    use_gradient_checkpointing=True
)

model = FastVisionModel.get_peft_model(
    model,
    fine_tune_vision_layers=False,
    fine_tune_language_layers=True,
    fine_tune_attention_modules=True,
    finetune_mlp_modules=True,

    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",

    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA RTX A4000. Num GPUs = 1. Max memory: 14.615 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu129. CUDA: 8.6. CUDA Toolkit: 12.9. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

[accelerate.big_modeling|WARNING]Some parameters are on the meta device because they were offloaded to the cpu.
[unsloth_zoo.log|WARNING]Unsloth: Failed to register input-embedding hook for `model.base_model.model.model.visual`: `get_input_embeddings` not auto‑handled for Qwen3_5VisionModel; please override in the subclass.. Falling back to pre-forward hook.


In [4]:
from datasets import load_dataset

dataset = load_dataset('lavita/MedQuAD')
dataset = dataset['train']
dataset,dataset[0]

(Dataset({
     features: ['document_id', 'document_source', 'document_url', 'category', 'umls_cui', 'umls_semantic_types', 'umls_semantic_group', 'synonyms', 'question_id', 'question_focus', 'question_type', 'question', 'answer'],
     num_rows: 47441
 }),
 {'document_id': '0000559',
  'document_source': 'GHR',
  'document_url': 'https://ghr.nlm.nih.gov/condition/keratoderma-with-woolly-hair',
  'category': None,
  'umls_cui': 'C0343073',
  'umls_semantic_types': 'T047',
  'umls_semantic_group': 'Disorders',
  'synonyms': 'KWWH',
  'question_id': '0000559-1',
  'question_focus': 'keratoderma with woolly hair',
  'question_type': 'information',
  'question': 'What is (are) keratoderma with woolly hair ?',
  'answer': 'Keratoderma with woolly hair is a group of related conditions that affect the skin and hair and in many cases increase the risk of potentially life-threatening heart problems. People with these conditions have hair that is unusually coarse, dry, fine, and tightly curled. 

## Dataset Processing - Prepare For Fine Tuning

#### Prompt Formatting
Build the data dictionary so that every fine tuning prompt passed to LLM is in the format expected by it (**ChatML**). The chat template is applied to the tokenizer so that the tokenizer object is permenantly modified. This allows the tokenizer to know how to use Qwen specific tags `<|im_start|>` and `<|im_end|>`

The ChatML template must be bound to the tokenizer first before mapping the dataset.

#### Train On Responses Only
This is a technique that ensures **Completion-Only Training** for the benchmarking protocol. It ensures the model's loss function only optimizes for generating the medical answer rather than wasting computational power learning how to memorize the user's questions

#### Dataset Split
The dataset needs to be split into 90-10 ratio where 90% is split for training and the remaining 10% for testing. The data processing steps need to be applied to both the train and test splits. If the processing is only applied on the train split, the `SFTTrainer` will be looking for specific ChatML tags in the test set when calculating the **validation loss** and since the tags are not present, incorrect validation loss will be reported 

In [6]:
from unsloth.chat_templates import get_chat_template
# Bind the template to tokenizer
tokenizer = get_chat_template(
    tokenizer,
    chat_template='qwen3-instruct'
)

print(f"rows before cleaning: {len(dataset)}")

def remove_empty_rows(example):
    # Check if the question or answer is none
    if example['question'] is None or example['answer'] is None:
        return False
    
    # Check if they are just empty strings or whitespace
    if len(str(example['question']).strip()) == 0 or len(str(example['answer']).strip()) == 0:
        return False
    
    return True

clean_dataset = dataset.filter(remove_empty_rows)

print(f"rows after cleaning: {len(clean_dataset)}")

# Split the cleaned dataset
split_dataset = clean_dataset.train_test_split(test_size=0.1, seed=42)

# Define the formatting function
def format_medquad_prompts(examples):
    texts = []
    for question, answer in zip(examples['question'], examples['answer']):
        messages = [
            {'role': 'user', 'content': question},
            {'role': 'assistant', 'content': answer}
        ]
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        texts.append(text)
    return {'text': texts}

print("applying ChatML formatting...")

formatted_dataset = split_dataset.map(format_medquad_prompts, batched=True)

rows before cleaning: 47441
rows after cleaning: 16407
applying ChatML formatting...


In [7]:
formatted_dataset, formatted_dataset['train'], formatted_dataset['test']

(DatasetDict({
     train: Dataset({
         features: ['document_id', 'document_source', 'document_url', 'category', 'umls_cui', 'umls_semantic_types', 'umls_semantic_group', 'synonyms', 'question_id', 'question_focus', 'question_type', 'question', 'answer', 'text'],
         num_rows: 14766
     })
     test: Dataset({
         features: ['document_id', 'document_source', 'document_url', 'category', 'umls_cui', 'umls_semantic_types', 'umls_semantic_group', 'synonyms', 'question_id', 'question_focus', 'question_type', 'question', 'answer', 'text'],
         num_rows: 1641
     })
 }),
 Dataset({
     features: ['document_id', 'document_source', 'document_url', 'category', 'umls_cui', 'umls_semantic_types', 'umls_semantic_group', 'synonyms', 'question_id', 'question_focus', 'question_type', 'question', 'answer', 'text'],
     num_rows: 14766
 }),
 Dataset({
     features: ['document_id', 'document_source', 'document_url', 'category', 'umls_cui', 'umls_semantic_types', 'umls_semantic_g

In [8]:
formatted_dataset['train'][0], formatted_dataset['test'][0]

({'document_id': '0003947',
  'document_source': 'GARD',
  'document_url': 'https://rarediseases.info.nih.gov/gard/1357/mental-retardation-hypotonic-facies-syndrome-x-linked-1',
  'category': None,
  'umls_cui': 'C0025362|C2931183|C0206064|C0241938',
  'umls_semantic_types': 'T048|T046|T047',
  'umls_semantic_group': 'Disorders',
  'synonyms': 'MRXHF1|Chudley syndrome 1|Chudley mental retardation syndrome|Chudley Lowry Hoar syndrome|Smith Fineman Myers syndrome 1',
  'question_id': '0003947-1',
  'question_focus': 'Mental retardation-hypotonic facies syndrome X-linked, 1',
  'question_type': 'symptoms',
  'question': 'What are the symptoms of Mental retardation-hypotonic facies syndrome X-linked, 1 ?',
  'answer': 'What are the signs and symptoms of Mental retardation-hypotonic facies syndrome X-linked, 1? The Human Phenotype Ontology provides the following list of signs and symptoms for Mental retardation-hypotonic facies syndrome X-linked, 1. If the information is available, the tabl

In [10]:
import numpy as np
import pandas as pd

# 1. Calculate token lengths (CPU only)
# We use .encode() to get the actual count the model sees
token_lengths = [len(tokenizer.tokenizer.encode(x)) for x in formatted_dataset['train']['text']]
lengths_arr = np.array(token_lengths)

# 2. Compute the Statistical "Hard Truths"
stats = {
    "Total Rows": len(lengths_arr),
    "Minimum Length": np.min(lengths_arr),
    "Maximum Length": np.max(lengths_arr),
    "Mean Length": round(np.mean(lengths_arr), 2),
    "Median (50th)": np.median(lengths_arr),
    "95th Percentile": np.percentile(lengths_arr, 95),
    "99th Percentile": np.percentile(lengths_arr, 99),
}

# 3. Check for Truncation at your 2048 limit
truncated_count = np.sum(lengths_arr > 2048)
percent_truncated = (truncated_count / len(lengths_arr)) * 100

print("--- MEDQUAD DATASET PROFILE ---")
for key, value in stats.items():
    print(f"{key:20}: {value}")

print("-" * 31)
print(f"ROWS EXCEEDING 2048: {truncated_count}")
print(f"TRUNCATION RATE    : {percent_truncated:.4f}%")

--- MEDQUAD DATASET PROFILE ---
Total Rows          : 14766
Minimum Length      : 20
Maximum Length      : 5726
Mean Length         : 297.08
Median (50th)       : 208.0
95th Percentile     : 778.0
99th Percentile     : 1661.7500000000018
-------------------------------
ROWS EXCEEDING 2048: 104
TRUNCATION RATE    : 0.7043%


## Fine Tuning Configuration

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    tokenizer=tokenizer,
    train_dataset = formatted_dataset['train'],
    eval_dataset = formatted_dataset['test'],
    args=SFTConfig(
        dataset_text_field='text',

        # Architecture configuration
        max_seq_length=2048,
        dataset_num_proc= 2, # Uses multiple CPU cores to speed up data loading

        # Memory and speed control
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        optim='adamw_8bit',
        bf16=True, # Activates A4000 hardware acceleration

        # Duration control
        num_train_epochs=3,

        # Learning rate and scheduling
        learning_rate=2e-4,
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        warmup_ratio=0.1,

        # Logging 
        logging_steps=10,
        report_to="wandb",
        seed=3407,

        # Auto-revert to best weights
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        greater_is_better=False,
        save_total_limit=2,

        # Ensure checkpointing to prevent catastrophic failure
        output_dir="medquad_checkpoints",
        per_device_eval_batch_size=1,
        eval_accumulation_steps=10,

        # Syncing Eval and Save to happen every 100 steps for frequent backups
        eval_strategy="steps",
        eval_steps=100,
        save_strategy="steps",
        save_steps=100
    )
)

trainer = train_on_responses_only(
    trainer,
    instruction_part='<|im_start|>user\n',
    response_part='<|im_start|>assistant\n'
)

In [ ]:
import os
import wandb

# Initialize wandb to track performance
wandb.init(
    project=os.environ['WANDB_PROJECT'],
    name='qwen-4b-medquad',
    tags=['qwen', '16-bit', 'medquad'],
    config=trainer.args
)

# START TRAINING
print("\033[1;31mstarting training loop...\033[1;31m")
trainer_stats = trainer.train()

# LOCAL SAVE
local_save_path = "models/qwen-3.5-4b-medquad-lora"
print(f"\033[1;31msaving locally to path {local_save_path}\033[1;31m")

model.save_pretrained(local_save_path)
tokenizer.save_pretrained(local_save_path)

# PUSH TO HUGGING FACE
hf_token = os.environ.get("HF_API_KEY")

try:
    hf_repo_name = "loknezmonzter/qwen-3.5-4b-medquad-lora"
    print("\033[1;31mpushing to huggingface repo {hf_repo_name}\033[1;31m")

    # Merge LoRA adapters to base model
    # Creates a single, unified 16-bit precision model (fine tuned)
    model.push_to_hub_merged(
        hf_repo_name,
        tokenizer,
        save_method="merged_16bit",
        token=hf_token
    )

    print("\033[1;31mcloud sync complete!\033[1;31m")
except Exception as e:
    print(f"\033[1;31m[ERR]error while pushing: {e}\033[1;31m")

In [2]:
print(tokenizer)

NameError: name 'tokenizer' is not defined